In [ ]:
import pandas as pd
import numpy as np

In [ ]:
link=pd.read_csv("link.csv")
movies=pd.read_csv("movie.csv")
ratings=pd.read_csv("rating.csv")
tag=pd.read_csv("tag.csv")




In [ ]:
genome_scores=pd.read_csv("genome_scores.csv")
genome_tags=pd.read_csv("genome_tags.csv")

In [ ]:
merged_df=pd.merge(ratings,movies,on='movieId')#to build a merged 

In [ ]:
print(link.head())
print(movies.head())
print(ratings.head())
print(tag.head())
print(genome_scores.head())
print(genome_tags.head())

In [ ]:
print(merged_df)

In [ ]:
tag_agg=tag.groupby('movieId')['tag'].apply(lambda x:' '.join(x.dropna())).reset_index()

In [ ]:
print(tag_agg)

In [ ]:
df=genome_scores.pivot(index='movieId',columns='tagId',values='relevance')

In [ ]:
print(df)


In [ ]:
print(type(df))
print(df.index)

In [ ]:
print(merged_df.shape)
print(tag_agg.shape)
print(df.shape)

In [ ]:
merged_df.drop_duplicates(inplace=True)
tag_agg.drop_duplicates(inplace=True)
df.drop_duplicates(inplace=True)
print(merged_df.shape)
print(tag_agg.shape)
print(df.shape)
#no duplicate represent

In [ ]:
merged_df.dropna(inplace=True)
tag_agg.dropna(inplace=True)
print(merged_df.shape)
print(tag_agg.shape)

In [ ]:
new_df=pd.merge(movies,tag_agg,on='movieId',how='left')

In [ ]:
new_df['tag']=new_df['tag'].fillna(' ')

In [ ]:
print(link.head())
print(movies.head())
print(ratings.head())
print(tag.head())
print(genome_scores.head())
print(genome_tags.head())

In [ ]:
movies.shape

In [ ]:
movies['genres'].shape

In [ ]:
new_df['genres']=new_df['genres'].fillna(' ')

In [ ]:
new_df['content']=new_df['genres']+' '+new_df['tag']

In [ ]:
print(new_df)

In [ ]:
#this was content based recommendation system based on content
from sklearn.feature_extraction.text import TfidfVectorizer
t=TfidfVectorizer(max_features=5000,ngram_range=(1,2))
tdidf_matrix=t.fit_transform(new_df['content'])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cos_matrix=cosine_similarity(tdidf_matrix)
print(cos_matrix)
print(cos_matrix.shape)



In [ ]:
indices=pd.Series(data=new_df.index,index=new_df['title']).drop_duplicates()

In [ ]:
print(indices)

In [ ]:
def recommend(movie_name,top_n=5):
    if movie_name not in indices:
        return "Movie not found!"
    idx=indices[movie_name]
    similarity_scores=list(enumerate(cos_matrix[idx]))
    similarity_scores=sorted(similarity_scores,key=lambda x:x[1],reverse=True)
    top_5=[x[0] for x in similarity_scores[1:top_n+1]]
    return new_df['title'].iloc[top_5]
    
    

In [ ]:
#now this was content based recommendation system based on df(tagid,movie_id,and thier relevance)

In [ ]:
recommend("Toy Story (1995)")

In [ ]:
#convert the df matrix into item-iitem similarity matrix
from  sklearn.metrics.pairwise import cosine_similarity
item_matrix=cosine_similarity(df.fillna(0))

In [ ]:
print(item_matrix)

In [ ]:
print(item_matrix.shape)
print(type(item_matrix))

In [ ]:
l=[]
for i in range(0,len(df)):
    l.append(i)
    
    


In [ ]:
s=pd.Series(data=new_df['title'],index=new_df['movieId'])
series=pd.Series(data=l,index=s[df.index])

In [ ]:
print(series)

In [ ]:
def recommend(movie_name,top_n=5):
    if movie_name not in series:
        return "Movie not found!"
    idx=series[movie_name]
    similarity_scores=list(enumerate(item_matrix[idx]))
    similarity_scores=sorted(similarity_scores,key=lambda x:x[1],reverse=True)
    top_5=[x[0] for x in similarity_scores[1:top_n+1]]
    return series.index[top_5]
    
    

In [ ]:
recommend('Jumanji (1995)')

In [ ]:
ratings.duplicated(subset=['userId','movieId']).sum()

In [ ]:
ratings=ratings[['userId','movieId','rating']]
print(ratings)
ratings.dropna()

In [ ]:
user_count=ratings['userId'].value_counts()
movie_count=ratings['movieId'].value_counts()
print(user_count)
print(movie_count)

In [ ]:
'''ratings=ratings[ratings['movieId'].isin(user_count[user_count>1000].index)]
ratings=ratings[ratings['userId'].isin(user_count[user_count>1000].index)]'''




In [ ]:
'''user_similarity=ratings.pivot_table(index='userId',columns='movieId',values='rating')'''

In [ ]:
'''from sklearn.metrics.pairwise import cosine_similarity
user_similarity_matrix=cosine_similarity(user_similarity)'''

In [ ]:
'''user_similarity_matrix=pd.DataFrame(user_similarity_matrix,index=user_similarity.columns,coluns=user_similarity.columns)'''

In [ ]:
'''def recommendv(movie_name,top_n=5):
     idx=movies[movies['title']==movie_name]['movieId'].values()
     if idx.empty:
          return "there is no movies"
      idx=idx[0]
     similarity_scores=user_similarity_matrix[idx].sort_values(ascending=False)[1:top_n+1]
     return movies[movies['movieId'].isin(similarity_scores.index)]['title']'''
       

In [ ]:
from sklearn.decomposition import PCA
pca=PCA(n_components=50)
genome_pca=pca.fit_transform(df)

In [ ]:
print(genome_pca.explained_variance_ratio_.sum())

In [ ]:
from sklearn.manifold import TSNE
t=TSNE(n_components=2,perplexity=30,random_state=42)
genome_result=t.fit_transform(genome_pca[:2000])

In [ ]:
from matplotlib.pyplot as plt
plt.scatter(genome_result[:,0],genome_result[:,1])
plt.title('Visualization')
plt.show()

In [ ]:
from sklearn.cluster import KMeans
kmeans=KMeans(n_clusters=10,random_state=42)
clusters=kmeans.fit_transform(genome_pca)

In [ ]:
#each cluster represent similar pattern of tag relevance
movies_subset=movies[movies['movieId'].isin(df.index)]
movies_subset['cluster']=clusters

In [ ]:
print(movies[['title','cluster']].head()

In [ ]:
def recommend(movie_name):
    movie_id=movies[movies['title']==movie_name]['movieId'].values[0]
    cluster_id=movies_subset[movies_subset['movieId']==movie_id]['cluster'].values[0]
    return movies_subset[movies_subset['cluster']==cluster_id]]['title'].head(10)

In [ ]:
from sklearn.metrics  import silhouette_score
score=silhoutte_score(genome_pca,clusters)
print("silhoutte score:",score)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
scores=[]

for k in range(2,15):
    kmeans=KMeans(n_clusters=k,random_state=42)
    clusters=kmeans.fit_transform(genome_pca)
    score=silhoutte_score(clusters,genome_pca)
    scores.append(score)
    
    print(f"{k}, score={score}")
    
    

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(2,15),scores)
plt.xlabel("Number of clusters")
plt.ylabel("sihoutte score")
plt.title("Sihouuette analysis")
plt.show()


In [ ]:
interia=[]
from sklearn.cluster import KMeans
for k in range(1,15):
    kmeans=KMeans(n_clusters=k,random_state=42)
    kmeans.fit(genome_pca)
    interia=interia.append(kmeans.interia_)
    


In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(1,15),interia)
plt.xlabel("k")
plt.ylabel("interia")
plt.show()
